# CrimeMapAI — Dataset Compatibility Audit

This notebook is the bridge between **data understanding** and **preprocessing**. It does **not** merge datasets automatically.

Primary question: **Can the 2017–2022 and 2024 district-level cybercrime datasets be used as one longitudinal modelling dataset?**

The audit checks grain, geography, schema overlap, candidate semantic matches, and the 2023/classification break. Structural column similarity is not treated as semantic equivalence.

In [10]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

from crimemapai.data.compatibility import (
    compatibility_decision,
    cross_dataset_key_audit,
    geography_overlap,
    normalized_column_candidates,
)

## 1. Load the two raw datasets

Replace these paths with the actual raw CSV locations used by the team. Keep the files separate until the audit is complete.

In [11]:
RAW_DIR = Path("../data/raw")

OLD_PATH = RAW_DIR / "districtwise-cyber-crimes-2017-onwards.csv"
NEW_PATH = RAW_DIR / "districtwise-cyber-crimes-2024-onwards.csv"

assert OLD_PATH.exists(), f"Raw dataset not found: {OLD_PATH}"
assert NEW_PATH.exists(), f"Raw dataset not found: {NEW_PATH}"

# Canonical dataframe variables used throughout this notebook.
old = pd.read_csv(OLD_PATH, low_memory=False)
new = pd.read_csv(NEW_PATH, low_memory=False)

print(f"2017–2022: {old.shape}")
print(f"2024: {new.shape}")

2017–2022: (5322, 48)
2024: (964, 56)


## 2. Grain audit

In [12]:
candidate_keys = [
    ['state_code', 'district_code', 'year'],
    ['state_code', 'district_code', 'year', 'registration_circles'],
    ['district_name', 'year'],
]

display(cross_dataset_key_audit(old, new, candidate_keys))

,keys,left_rows,right_rows,left_unique_ratio,right_unique_ratio,left_duplicate_key_rows,right_duplicate_key_rows,left_null_key_rows,right_null_key_rows,common_keys,left_only_keys,right_only_keys,relationship,safe_1_to_1
0,state_code + district_code + year,5322,964,0.816798,0.799793,1571,302,0,0,0,4347,771,M:M,False
1,state_code + district_code + year + registrati...,5322,964,1.000000,1.000000,0,0,0,0,0,5322,964,1:1,True
2,district_name + year,5322,964,0.814543,0.796680,1607,308,0,0,0,4335,768,M:M,False


### Interpretation rule

The modelling grain must be explicitly frozen before preprocessing. `id` is a row identifier candidate, not a longitudinal entity key, unless the source documentation proves otherwise.

## 3. Geography compatibility

In [13]:
geo = geography_overlap(old, new, ['state_code', 'district_code'])
display(pd.DataFrame([geo]))

,left_unique_geographies,right_unique_geographies,common_geographies,left_only_geographies,right_only_geographies
0,753,771,748,5,23


## 4. Exact and normalized schema overlap

In [14]:
exact_common = [c for c in old.columns if c in set(new.columns)]
print(f'Exact common columns: {len(exact_common)}')
display(pd.DataFrame({'exact_common_column': exact_common}))

normalized = normalized_column_candidates(old, new)
display(normalized.sort_values(['exact_match', 'normalized_name'], ascending=[False, True]))

Exact common columns: 20


,exact_common_column
0,id
1,year
2,state_name
3,state_code
4,district_name
5,district_code
6,registration_circles
7,tampering_computer_source_documents
8,ransom_ware
9,offences_other_than_ransom_ware


,normalized_name,left_column,right_column,exact_match
0,cyber_blackmailing_threatening,cyber_blackmailing_threatening,cyber_blackmailing_threatening,True
1,cyber_terrorism,cyber_terrorism,cyber_terrorism,True
2,data_theft,data_theft,data_theft,True
3,defamation_morphing,defamation_morphing,defamation_morphing,True
4,district_code,district_code,district_code,True
5,district_name,district_name,district_name,True
6,fake_profile,fake_profile,fake_profile,True
7,id,id,id,True
8,identity_theft,identity_theft,identity_theft,True
9,offences_other_than_ransom_ware,offences_other_than_ransom_ware,offences_other_than_ransom_ware,True


## 5. Indicator crosswalk — manual semantic review

Use these statuses:

- `EXACT`: same source-defined meaning.
- `CANDIDATE`: likely equivalent, but source definition must be verified.
- `SEMANTICALLY_UNCERTAIN`: similar label but potentially different legal/scope definition.
- `NEW_IN_2024`: no defensible older equivalent identified.
- `REMOVED_OR_REPLACED`: older field has no directly compatible 2024 field.

Do not promote a `CANDIDATE` mapping to `EXACT` based only on the column name.

In [15]:
crosswalk = pd.DataFrame([
    ('cyber_blackmailing_threatening', 'cyber_blackmailing_threatening', 'EXACT'),
    ('cyber_terrorism', 'cyber_terrorism', 'EXACT'),
    ('data_theft', 'data_theft', 'EXACT'),
    ('identity_theft', 'identity_theft', 'EXACT'),
    ('online_banking_fraud', 'online_banking_fraud', 'EXACT'),
    ('other_frauds', 'other_frauds', 'EXACT'),
    ('ransom_ware', 'ransom_ware', 'EXACT'),
    ('atms_fraud', 'fraud_atms', 'CANDIDATE'),
    ('credit_card_debit_card_fraud', 'credit_card_debit', 'CANDIDATE'),
    ('copy_right_act', 'copy_right', 'CANDIDATE'),
    ('currency_counterfeiting', 'counterfeiting_currency', 'CANDIDATE'),
    ('stamps_counterfeiting', 'counterfeiting_stamps', 'CANDIDATE'),
    ('cheating_by_personation_by_using_computer_resource', 'cheating_personation_using_computer_resource', 'CANDIDATE'),
    ('forgery', 'offences_ipc_bns_it_forgery', 'SEMANTICALLY_UNCERTAIN'),
    ('abetment_of_suicide_online', 'abetment_suicide_107_108_bns_305', 'SEMANTICALLY_UNCERTAIN'),
    ('other_offences', 'other_offences_bns_ipc_it', 'SEMANTICALLY_UNCERTAIN'),
] , columns=['old_column', 'new_column', 'status'])
display(crosswalk)

,old_column,new_column,status
0,cyber_blackmailing_threatening,cyber_blackmailing_threatening,EXACT
1,cyber_terrorism,cyber_terrorism,EXACT
2,data_theft,data_theft,EXACT
3,identity_theft,identity_theft,EXACT
4,online_banking_fraud,online_banking_fraud,EXACT
5,other_frauds,other_frauds,EXACT
6,ransom_ware,ransom_ware,EXACT
7,atms_fraud,fraud_atms,CANDIDATE
8,credit_card_debit_card_fraud,credit_card_debit,CANDIDATE
9,copy_right_act,copy_right,CANDIDATE


## 6. Temporal and legal classification break

Current supplied datasets cover 2017–2022 and 2024. Therefore 2023 is absent from these files. The project must not describe the resulting panel as an uninterrupted 2017–2024 series unless 2023 is separately sourced.

The 2024 schema also contains IPC/BNS-specific fields. This means longitudinal modelling of affected indicators requires an explicit source-backed harmonization/crosswalk.

In [16]:
decision = compatibility_decision(
    has_temporal_gap=True,
    classification_change=True,
    grain_compatible=False,  # replace after the grain audit is verified
    geography_compatible=False,  # replace after the geography audit is verified
)
display(Markdown(f'### Current conservative decision: **{decision}**'))

### Current conservative decision: **DO_NOT_APPEND_YET**

## 7. Freeze the data strategy

Before opening `02_data_preprocessing.ipynb`, record one explicit decision:

1. **Longitudinal append after harmonization** — only if geography, grain, and indicator definitions are defensibly compatible.
2. **Separate-era modelling** — keep 2024 separate if classification changes make direct target continuity unreliable.
3. **Restricted common-indicator panel** — model only indicators with source-verified equivalence.

The chosen strategy must be written into `progress.md` and the final report.